# 👁️ Taller Semana 09 — Ojos Digitales: Visión Artificial

**Objetivo:** Entender los fundamentos de la percepción visual artificial mediante imágenes en escala de grises, filtros y detección básica de bordes usando OpenCV.

---

### 📋 Contenidos
1. Instalación y librerías
2. Carga y visualización de imagen
3. Conversión a escala de grises
4. Filtros convolucionales (Blur y Sharpening)
5. Detección de bordes — Sobel (X e Y)
6. Detección de bordes — Laplaciano
7. Comparación visual entre métodos
8. ⭐ Bonus: Sliders interactivos con `ipywidgets`

---
## 1. 📦 Instalación y librerías

In [ ]:
# OpenCV ya viene instalado en Colab, pero verificamos
!pip install opencv-python-headless --quiet
!pip install ipywidgets --quiet

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from google.colab import files
from IPython.display import display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, IntSlider, Dropdown

# Configuración general de matplotlib
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.titleweight'] = 'bold'

print('✅ Librerías cargadas correctamente')
print(f'   OpenCV versión: {cv2.__version__}')
print(f'   NumPy versión:  {np.__version__}')

---
## 2. 🖼️ Carga de la imagen

Sube el archivo `kitty.jpg` cuando se lo solicite la celda siguiente.

In [ ]:
# ── Subir imagen desde tu equipo ──────────────────────────────────────────────
print('📂 Selecciona el archivo kitty.jpg para subirlo...')
uploaded = files.upload()

# Detectar el nombre del archivo subido
img_name = list(uploaded.keys())[0]
print(f'\n✅ Archivo recibido: {img_name}')

In [ ]:
# ── Leer y mostrar la imagen original ─────────────────────────────────────────
img_bgr = cv2.imread(img_name)                   # OpenCV lee en BGR
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)  # Convertir a RGB para matplotlib

print(f'📐 Dimensiones: {img_rgb.shape[1]} x {img_rgb.shape[0]} px  |  Canales: {img_rgb.shape[2]}')
print(f'🎨 Tipo de dato: {img_rgb.dtype}  |  Rango: [{img_rgb.min()} - {img_rgb.max()}]')

plt.figure(figsize=(7, 5))
plt.imshow(img_rgb)
plt.title('🐱 Imagen Original — Color (RGB)')
plt.axis('off')
plt.tight_layout()
plt.show()

---
## 3. ⬛ Conversión a Escala de Grises

OpenCV utiliza la fórmula estándar de luminancia para convertir a gris:

> **Gray = 0.114·B + 0.587·G + 0.299·R**

Esta ponderación respeta la sensibilidad del ojo humano a cada color.

In [ ]:
# ── Convertir a escala de grises ───────────────────────────────────────────────
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].imshow(img_rgb)
axes[0].set_title('Original — Color (RGB)')
axes[0].axis('off')

axes[1].imshow(img_gray, cmap='gray')
axes[1].set_title('Escala de Grises')
axes[1].axis('off')

plt.suptitle('Paso 1 — Conversión a Escala de Grises', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\n📊 La imagen pasó de {img_rgb.shape} → {img_gray.shape}')
print('   (Se eliminó la dimensión de canales: ahora cada píxel es 1 valor 0–255)')

---
## 4. 🌀 Filtros Convolucionales

La **convolución** desliza un kernel (matriz pequeña) sobre la imagen y calcula la suma ponderada de los píxeles vecinos. El kernel define el efecto:

| Kernel | Efecto |
|--------|--------|
| Promedio / Gaussiano | Suavizado (blur) |
| Realce central | Nitidez (sharpening) |

In [ ]:
# ── Definir kernels manualmente ────────────────────────────────────────────────

# Kernel de Blur (promedio 5×5)
kernel_blur = np.ones((5, 5), dtype=np.float32) / 25

# Kernel de Sharpening
kernel_sharp = np.array([[ 0, -1,  0],
                          [-1,  5, -1],
                          [ 0, -1,  0]], dtype=np.float32)

# Aplicar filtros
img_blur    = cv2.filter2D(img_gray, -1, kernel_blur)
img_sharp   = cv2.filter2D(img_gray, -1, kernel_sharp)

# También usar filtros nativos de OpenCV para comparar
img_gauss   = cv2.GaussianBlur(img_gray, (5, 5), sigmaX=1)

print('✅ Filtros aplicados')
print(f'   kernel_blur  → {kernel_blur.shape}   suma={kernel_blur.sum():.2f}')
print(f'   kernel_sharp → {kernel_sharp.shape}  suma={kernel_sharp.sum():.2f}')

In [ ]:
# ── Visualización de filtros ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

imagenes = [img_gray, img_blur, img_gauss, img_sharp]
titulos  = ['Original (Gris)', 'Blur (kernel 5×5)', 'Blur Gaussiano', 'Sharpening']

for ax, img, titulo in zip(axes, imagenes, titulos):
    ax.imshow(img, cmap='gray')
    ax.set_title(titulo)
    ax.axis('off')

plt.suptitle('Paso 2 — Filtros Convolucionales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. 📏 Detección de Bordes — Filtro de Sobel

El operador **Sobel** calcula la derivada parcial de la imagen en cada dirección. Detecta bordes midiendo el cambio de intensidad:

$$G_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix} * I \qquad G_y = \begin{bmatrix}-1&-2&-1\\0&0&0\\1&2&1\end{bmatrix} * I$$

$$G = \sqrt{G_x^2 + G_y^2}$$

In [ ]:
# ── Aplicar Sobel en X, Y y magnitud total ─────────────────────────────────────
# Usamos img_gauss (suavizada) para reducir el ruido antes de detectar bordes

sobel_x = cv2.Sobel(img_gauss, cv2.CV_64F, dx=1, dy=0, ksize=3)  # Bordes verticales
sobel_y = cv2.Sobel(img_gauss, cv2.CV_64F, dx=0, dy=1, ksize=3)  # Bordes horizontales

# Magnitud del gradiente
sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)

# Normalizar a 0–255 para visualización
sobel_x_vis   = cv2.convertScaleAbs(sobel_x)
sobel_y_vis   = cv2.convertScaleAbs(sobel_y)
sobel_mag_vis = cv2.convertScaleAbs(sobel_mag)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

datos = [(img_gauss,     'Base (Gaussiano)'),
         (sobel_x_vis,   'Sobel — Eje X\n(Bordes verticales)'),
         (sobel_y_vis,   'Sobel — Eje Y\n(Bordes horizontales)'),
         (sobel_mag_vis, 'Sobel — Magnitud\n√(Gx² + Gy²)')]

for ax, (img, titulo) in zip(axes, datos):
    ax.imshow(img, cmap='gray')
    ax.set_title(titulo, pad=8)
    ax.axis('off')

plt.suptitle('Paso 3 — Detección de Bordes: Operador Sobel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Sobel X detecta cambios horizontales de intensidad → bordes verticales')
print('💡 Sobel Y detecta cambios verticales de intensidad   → bordes horizontales')

---
## 6. 🔵 Detección de Bordes — Filtro Laplaciano

El **Laplaciano** usa la segunda derivada de la imagen. Es más sensible al ruido pero detecta bordes en todas las direcciones simultáneamente:

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

Kernel típico:
$$L = \begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix}$$

In [ ]:
# ── Aplicar Laplaciano ─────────────────────────────────────────────────────────
laplacian_raw = cv2.Laplacian(img_gauss, cv2.CV_64F, ksize=3)
laplacian_vis = cv2.convertScaleAbs(laplacian_raw)

# También probamos con ksize=5 para comparar
laplacian_k5 = cv2.Laplacian(img_gauss, cv2.CV_64F, ksize=5)
laplacian_k5_vis = cv2.convertScaleAbs(laplacian_k5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_gauss, cmap='gray')
axes[0].set_title('Base (Gaussiano)')
axes[0].axis('off')

axes[1].imshow(laplacian_vis, cmap='gray')
axes[1].set_title('Laplaciano (ksize=3)')
axes[1].axis('off')

axes[2].imshow(laplacian_k5_vis, cmap='gray')
axes[2].set_title('Laplaciano (ksize=5)')
axes[2].axis('off')

plt.suptitle('Paso 4 — Detección de Bordes: Operador Laplaciano', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Un ksize mayor captura bordes más gruesos pero pierde detalle fino')

---
## 7. 🔬 Comparación Visual entre Métodos

Reunimos todos los detectores de bordes en una sola figura para comparar directamente sus resultados.

In [ ]:
# ── Comparación completa ───────────────────────────────────────────────────────

# También incluimos Canny (bonus) para enriquecer la comparación
canny = cv2.Canny(img_gauss, threshold1=50, threshold2=150)

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.1)

resultados = [
    (img_rgb,         'Original (Color)',          'gray',   False),
    (img_gray,        'Escala de Grises',           'gray',   True ),
    (img_blur,        'Blur (promedio 5×5)',         'gray',   True ),
    (img_sharp,       'Sharpening',                 'gray',   True ),
    (sobel_x_vis,     'Sobel X',                    'gray',   True ),
    (sobel_y_vis,     'Sobel Y',                    'gray',   True ),
    (sobel_mag_vis,   'Sobel Magnitud',             'gray',   True ),
    (laplacian_vis,   'Laplaciano',                 'gray',   True ),
]

for idx, (img, titulo, cmap, es_gray) in enumerate(resultados):
    ax = fig.add_subplot(gs[idx // 4, idx % 4])
    if es_gray:
        ax.imshow(img, cmap=cmap)
    else:
        ax.imshow(img)
    ax.set_title(titulo, pad=6)
    ax.axis('off')

fig.suptitle('📊 Comparación Visual — Todos los Métodos', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('comparacion_metodos.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Imagen guardada como comparacion_metodos.png')

In [ ]:
# ── Histogramas de intensidad para cada método ─────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 6))

histogramas = [
    (img_gray,        'Gris original'),
    (img_blur,        'Blur'),
    (img_sharp,       'Sharpening'),
    (img_gauss,       'Gaussiano'),
    (sobel_x_vis,     'Sobel X'),
    (sobel_y_vis,     'Sobel Y'),
    (sobel_mag_vis,   'Sobel Magnitud'),
    (laplacian_vis,   'Laplaciano'),
]

for ax, (img, titulo) in zip(axes.flat, histogramas):
    ax.hist(img.ravel(), bins=64, color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_title(titulo, fontsize=9)
    ax.set_xlim(0, 255)
    ax.set_yticks([])

plt.suptitle('Histogramas de Intensidad por Método', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⭐ Bonus — Sliders Interactivos con `ipywidgets`

> **Nota:** `cv2.createTrackbar` requiere ventanas nativas de OpenCV, que no funcionan en Colab. En su lugar usamos `ipywidgets`, que ofrece la misma funcionalidad de forma interactiva dentro del notebook.

Puedes ajustar en tiempo real:
- Tamaño del kernel de blur
- Tipo de detector de bordes
- Umbral para Canny

In [ ]:
# ── Widget interactivo de filtros ──────────────────────────────────────────────

def aplicar_filtro(tipo_filtro, kernel_size, canny_low, canny_high, sigma):
    """Aplica el filtro seleccionado con los parámetros del slider y muestra el resultado."""

    # Asegurar que kernel_size sea impar
    k = kernel_size if kernel_size % 2 == 1 else kernel_size + 1

    # Pre-suavizado gaussiano
    base = cv2.GaussianBlur(img_gray, (k, k), sigmaX=sigma)

    if tipo_filtro == 'Gaussian Blur':
        resultado = base
        titulo_res = f'Gaussian Blur  (k={k}, σ={sigma})'

    elif tipo_filtro == 'Blur (promedio)':
        kernel = np.ones((k, k), dtype=np.float32) / (k * k)
        resultado = cv2.filter2D(img_gray, -1, kernel)
        titulo_res = f'Blur Promedio  (k={k})'

    elif tipo_filtro == 'Sharpening':
        kernel_s = np.array([[ 0, -1,  0], [-1,  5, -1], [ 0, -1,  0]], dtype=np.float32)
        resultado = cv2.filter2D(img_gray, -1, kernel_s)
        titulo_res = 'Sharpening'

    elif tipo_filtro == 'Sobel X':
        s = cv2.Sobel(base, cv2.CV_64F, dx=1, dy=0, ksize=k if k <= 7 else 7)
        resultado = cv2.convertScaleAbs(s)
        titulo_res = f'Sobel X  (k={k if k<=7 else 7})'

    elif tipo_filtro == 'Sobel Y':
        s = cv2.Sobel(base, cv2.CV_64F, dx=0, dy=1, ksize=k if k <= 7 else 7)
        resultado = cv2.convertScaleAbs(s)
        titulo_res = f'Sobel Y  (k={k if k<=7 else 7})'

    elif tipo_filtro == 'Sobel Magnitud':
        sx = cv2.Sobel(base, cv2.CV_64F, dx=1, dy=0, ksize=3)
        sy = cv2.Sobel(base, cv2.CV_64F, dx=0, dy=1, ksize=3)
        resultado = cv2.convertScaleAbs(np.sqrt(sx**2 + sy**2))
        titulo_res = 'Sobel Magnitud'

    elif tipo_filtro == 'Laplaciano':
        lap = cv2.Laplacian(base, cv2.CV_64F, ksize=k if k <= 7 else 7)
        resultado = cv2.convertScaleAbs(lap)
        titulo_res = f'Laplaciano  (k={k if k<=7 else 7})'

    elif tipo_filtro == 'Canny':
        resultado = cv2.Canny(base, threshold1=canny_low, threshold2=canny_high)
        titulo_res = f'Canny  (low={canny_low}, high={canny_high})'

    # Mostrar
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].imshow(img_gray, cmap='gray')
    axes[0].set_title('Original (Gris)', pad=8)
    axes[0].axis('off')

    axes[1].imshow(resultado, cmap='gray')
    axes[1].set_title(titulo_res, pad=8)
    axes[1].axis('off')

    plt.suptitle('🎛️ Filtro Interactivo', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# ── Controles ──────────────────────────────────────────────────────────────────
interact(
    aplicar_filtro,
    tipo_filtro = Dropdown(
        options=['Gaussian Blur', 'Blur (promedio)', 'Sharpening',
                 'Sobel X', 'Sobel Y', 'Sobel Magnitud',
                 'Laplaciano', 'Canny'],
        value='Gaussian Blur',
        description='Filtro:'
    ),
    kernel_size = IntSlider(min=1, max=21, step=2, value=5,   description='Kernel:'),
    canny_low   = IntSlider(min=0, max=150, step=5, value=50,  description='Canny Low:'),
    canny_high  = IntSlider(min=50, max=300, step=10, value=150, description='Canny High:'),
    sigma       = IntSlider(min=0, max=10, step=1, value=1,    description='Sigma σ:'),
);

---
## ⭐ Bonus Extra — Detector Canny + Comparación Final

In [ ]:
# ── Canny: el detector de bordes más robusto ───────────────────────────────────
# Canny combina: suavizado → gradiente Sobel → supresión de no-máximos → umbralización doble

canny_edges = cv2.Canny(img_gauss, threshold1=50, threshold2=150)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

datos = [
    (img_gray,        'Original Gris'),
    (sobel_mag_vis,   'Sobel Magnitud'),
    (laplacian_vis,   'Laplaciano'),
    (canny_edges,     'Canny (50/150)'),
]

for ax, (img, titulo) in zip(axes, datos):
    ax.imshow(img, cmap='gray')
    ax.set_title(titulo, pad=8)
    ax.axis('off')

plt.suptitle('🔍 Comparación Final de Detectores de Bordes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Tabla de resumen
print('\n📋 RESUMEN DE MÉTODOS:')
print('─' * 62)
print(f'{"Método":<20} {"Derivada":<12} {"Sensible al ruido":<20} {"Dirección"}')
print('─' * 62)
print(f'{"Sobel X/Y":<20} {"1ª orden":<12} {"Media":<20} {"Separada X o Y"}')
print(f'{"Sobel Magnitud":<20} {"1ª orden":<12} {"Media":<20} {"Todas"}')
print(f'{"Laplaciano":<20} {"2ª orden":<12} {"Alta":<20} {"Todas"}')
print(f'{"Canny":<20} {"1ª orden":<12} {"Baja (robusto)":<20} {"Todas"}')
print('─' * 62)

---
## 🎓 Conclusiones

| Concepto | Lo que aprendimos |
|---|---|
| **Escala de grises** | Reduce 3 canales a 1 usando pesos de luminancia humana |
| **Convolución** | Operación matricial que aplica un kernel para transformar píxeles |
| **Blur** | Promedía vecinos → suaviza → elimina ruido |
| **Sharpening** | Refuerza el centro → resalta detalles |
| **Sobel** | Derivada 1ª → detecta cambios de intensidad por dirección |
| **Laplaciano** | Derivada 2ª → detecta bordes en todas las direcciones |
| **Canny** | Pipeline completo → el más preciso para bordes nítidos |

---
*Taller Semana 09 — Visión Artificial con OpenCV*